# Compiled ONNX SmolLM2 INT8 on PYNQ

Build and deployment instructions are in `README.md` (WSL) and `README.macos.md`. This demo uses shared `src/` code and the board's PYNQ package.

The package imports ONNX with ONNX-MLIR, partitions CPU/NPU work, and compiles CPU operators to ARM32. Physical acceptance passed all prefill profiles and three decode steps each. This notebook also completed 32 tokens matching the independent source ONNX reference on PYNQ-Z1.

Run in the board's root PYNQ kernel. Defaults are the notebook directory, `smollm-package/`, and `npu_matrix.bit`. Set `NPU_REPO`, `NPU_COMPILED_PACKAGE`, and `NPU_OVERLAY` before setup for another layout.


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("NPU_REPO", str(Path.cwd()))).expanduser().resolve()
PACKAGE = Path(os.environ.get("NPU_COMPILED_PACKAGE", str(REPO / "smollm-package"))).expanduser().resolve()
OVERLAY_PATH = Path(os.environ.get("NPU_OVERLAY", str(REPO / "npu_matrix.bit"))).expanduser().resolve()
for required_path in (REPO / "src/runtime/core/engine.py", PACKAGE / "manifest.json", OVERLAY_PATH, OVERLAY_PATH.with_suffix(".hwh")):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required deployment file is missing: {required_path}")
os.environ.setdefault("XILINX_XRT", "/usr")
sys.path.insert(0, str(REPO))
from src.runtime.core.engine import LLMEngine
from pynq import Overlay

In [ ]:
overlay = Overlay(str(OVERLAY_PATH))
engine = LLMEngine(PACKAGE, backend="pynq", overlay=overlay)
print(engine.meta.get("compiled_pipeline", "ONNX-MLIR -> CPU/NPU partition -> LLVM ARM + NPU ISA"))
print("Supported exact chat lengths:", engine.prefill_lengths)

## Edit the prompt
The complete chat token length must have a compiled prefill specialization. The default package supports 36, 40 and 42 tokens. Compile additional exact lengths for other prompts; generation stops at EOS or the context limit.

In [ ]:
prompt = "Write a paragraph about a neural processing unit (NPU)."
ids, text, stats = engine.generate(
    prompt, max_new_tokens=32, chat=True,
    on_token=lambda _, piece: print(piece, end="", flush=True),
)
print()
print(stats.as_dict())
print(engine.package.stats())

## Release board buffers
Run this when finished with the engine. Re-run setup and engine initialization before generating again.

In [ ]:
for buffer_name in ("weights", "io", "stage", "words"):
    allocated_buffer = getattr(engine.package, buffer_name, None)
    if allocated_buffer is not None and hasattr(allocated_buffer, "buffer"):
        allocated_buffer.buffer.freebuffer()
del allocated_buffer
del engine